# C5-neural-networks — Practice p17 — Solution

Rolling the vertex rows by $-1$ pairs every vertex $p$ with its successor $q$.  From $d=q-p$, column reversal plus the signs $(-d_2,d_1)$ produces every inward normal at once.  The bias is the negative component sum of $n\odot p$.

A vertex lies on its two incident edge boundaries, so two detector pre-activations are exactly zero.  The inclusive step makes both detectors fire, and convexity puts the vertex on the valid side of every other edge; consequently all five vertices are members.

In [ ]:
import numpy as np


def affine_layer(x, W, b):
    return (x[:, None, :] * W[None, :, :]).sum(axis=2) + b


def step_activation(z):
    return (z >= 0).astype(float)


SEED = 20260804
verts = np.array([[0.0, 0.0], [4.0, -1.0], [6.0, 2.0], [3.0, 5.0], [-1.0, 3.0]])
rng = np.random.default_rng(SEED)
pts = rng.uniform([-2, -2], [7, 6], (2000, 2))
edge_d = np.roll(verts, -1, axis=0) - verts
W_hull = np.stack([-edge_d[:, 1], edge_d[:, 0]], axis=1)
b_hull = -(W_hull * verts).sum(axis=1)


def region_indicator(points, W, b):
    detector_values = step_activation(affine_layer(points, W, b))
    m = W.shape[0]
    gate_W = np.ones((1, m))
    gate_b = np.array([-(m - 0.5)])
    return step_activation(affine_layer(detector_values, gate_W, gate_b))[:, 0]


centroid_label = float(region_indicator(verts.mean(axis=0, keepdims=True), W_hull, b_hull)[0])
vertex_labels = region_indicator(verts, W_hull, b_hull)
frac_in = float(region_indicator(pts, W_hull, b_hull).mean())
W_hull, b_hull, centroid_label, vertex_labels, frac_in

### Answer check

In [ ]:
assert np.allclose(W_hull, [[1.0, 4.0], [-3.0, 2.0], [-3.0, -3.0], [2.0, -4.0], [3.0, 1.0]], atol=1e-12)
assert np.allclose(b_hull, [0.0, 14.0, 24.0, 14.0, 0.0], atol=1e-12)
assert np.isclose(centroid_label, 1.0, atol=1e-12)
assert np.allclose(vertex_labels, np.ones(5), atol=1e-12)
assert np.isclose(frac_in, 0.372, atol=1e-12)